In [1]:
import numpy as np 
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

df = pd.read_csv(
    '/kaggle/input/datasets/kekavigi/earthquakes-in-indonesia/katalog_gempa_v2.tsv', 
    sep='\t', 
    low_memory=False
)

/kaggle/input/datasets/kekavigi/earthquakes-in-indonesia/katalog_gempa.csv
/kaggle/input/datasets/kekavigi/earthquakes-in-indonesia/katalog_gempa_v2.tsv
/kaggle/input/datasets/kekavigi/earthquakes-in-indonesia/katalog_sensor.tsv
/kaggle/input/datasets/kekavigi/earthquakes-in-indonesia/usgs.sqlite


In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.head())

(176384, 37)
['eventID', 'datetime', 'latitude', 'longitude', 'magnitude', 'mag_type', 'depth', 'phasecount', 'azimuth_gap', 'location', 'agency', 'datetimeFM', 'latFM', 'lonFM', 'magFM', 'magTypeFM', 'depthFM', 'phasecountFM', 'AzGapFM', 'scalarMoment', 'Mrr', 'Mtt', 'Mpp', 'Mrt', 'Mrp', 'Mtp', 'varianceReduction', 'doubleCouple', 'clvd', 'strikeNP1', 'dipNP1', 'rakeNP1', 'strikeNP2', 'dipNP2', 'rakeNP2', 'azgapFM', 'misfit']
       eventID                          datetime  latitude   longitude  \
0  bmg2008vkye  2008-11-01 00:31:25.143741+00:00 -0.604440   98.895531   
1  bmg2008vkzn  2008-11-01 01:13:15.738091+00:00 -3.386660  148.874180   
2  bmg2008vlag  2008-11-01 01:34:29.660856+00:00 -6.611860  129.387220   
3  bmg2008vlaj  2008-11-01 01:38:14.802129+00:00 -3.650586  127.990680   
4  bmg2008vlbt  2008-11-01 02:20:05.909515+00:00 -4.198925  128.097000   

   magnitude mag_type  depth  phasecount  azimuth_gap  \
0   2.989742      MLv     20           6    146.70738   
1   5.2081

Wilayah mana yang paling sering mengalami gempa, dan apakah polanya berubah dari tahun ke tahun?
Apakah ada hubungan antara kedalaman dan magnitudo?
Apakah jumlah gempa tercatat naik, dan apakah itu karena gempanya bertambah atau karena sensornya bertambah?

In [3]:
cols = ["eventID","datetime","latitude","longitude","magnitude","mag_type","depth","location"]
d = df[cols].copy()

In [4]:
print(d.shape)
print(d.columns.tolist())
print(d.head())

(176384, 8)
['eventID', 'datetime', 'latitude', 'longitude', 'magnitude', 'mag_type', 'depth', 'location']
       eventID                          datetime  latitude   longitude  \
0  bmg2008vkye  2008-11-01 00:31:25.143741+00:00 -0.604440   98.895531   
1  bmg2008vkzn  2008-11-01 01:13:15.738091+00:00 -3.386660  148.874180   
2  bmg2008vlag  2008-11-01 01:34:29.660856+00:00 -6.611860  129.387220   
3  bmg2008vlaj  2008-11-01 01:38:14.802129+00:00 -3.650586  127.990680   
4  bmg2008vlbt  2008-11-01 02:20:05.909515+00:00 -4.198925  128.097000   

   magnitude mag_type  depth                     location  
0   2.989742      MLv     20  Southern Sumatra, Indonesia  
1   5.208121       mb     51                 Bismarck Sea  
2   5.507549       mb     30                    Banda Sea  
3   3.539674      MLv      5             Seram, Indonesia  
4   2.424314      MLv      5                    Banda Sea  


In [5]:
print(d.isna().sum())
print(d["mag_type"].value_counts())

eventID         0
datetime        0
latitude        0
longitude       0
magnitude    1405
mag_type     1405
depth           0
location       28
dtype: int64
mag_type
M          74261
MLv        71671
mb         19776
Mw(mB)      5764
Mw          3046
ML           239
Mw(Mwp)       92
mB            85
Mwp           38
Mjma           7
Name: count, dtype: int64


In [6]:
d["datetime"] = pd.to_datetime(d["datetime"], format="ISO8601", utc=True)
d["tahun"] = d["datetime"].dt.year
print(d["tahun"].value_counts().sort_index())

tahun
2008      239
2009     4134
2010     7646
2011     6261
2012     8510
2013     5051
2014     7209
2015     7511
2016     7991
2017     8898
2018    12517
2019    13635
2020     9911
2021    12653
2022    12438
2023    13389
2024     9097
2025    21021
2026     8273
Name: count, dtype: int64


In [7]:
d[d["magnitude"].isna()]["mag_type"].isna().all()
pd.crosstab(d["tahun"], d["mag_type"])

mag_type,M,ML,MLv,Mjma,Mw,Mw(Mwp),Mw(mB),Mwp,mB,mb
tahun,,,,,,,,,,
2008,0,0,132,0,0,0,20,0,0,87
2009,394,0,2213,0,0,0,230,0,0,1297
2010,5839,0,1130,0,0,1,372,1,0,303
2011,5825,0,9,0,0,0,372,0,0,43
2012,6234,0,1408,0,3,1,322,2,9,521
2013,1055,0,2414,0,9,7,263,4,4,1281
2014,172,1,3915,0,4,7,328,6,15,2741
2015,124,0,4523,0,7,9,274,7,7,2515
2016,172,31,4791,0,87,5,266,2,12,2518


In [8]:
# 1. Apakah nilai magnitudo antar jenis kira-kira setara?
print(d.groupby("mag_type")["magnitude"].describe())

# 2. Saring wilayah Indonesia + buang magnitudo kosong
idn = d[(d["latitude"].between(-11, 6)) &
        (d["longitude"].between(94, 142)) &
        (d["magnitude"].notna())].copy()
print(len(idn))

# 3. Jumlah gempa per tahun hanya untuk magnitudo >= 4
print(idn[idn["magnitude"] >= 4]["tahun"].value_counts().sort_index())

            count      mean       std       min       25%       50%       75%  \
mag_type                                                                        
M         74261.0  3.685041  0.904560  0.638354  2.998676  3.585044  4.366315   
ML          239.0  2.882653  0.821148  0.829124  2.201890  2.899853  3.519582   
MLv       71671.0  3.298949  0.726372  0.191153  2.775941  3.283960  3.822082   
Mjma          7.0  3.531616  0.778028  2.454721  3.041550  3.384714  4.108078   
Mw         3046.0  4.861915  0.750812  2.650000  4.325201  4.744625  5.299055   
Mw(Mwp)      92.0  6.106542  0.679988  4.687536  5.743131  6.130543  6.456599   
Mw(mB)     5764.0  5.494663  0.637326  3.231831  5.067854  5.441441  5.848154   
Mwp          38.0  6.012205  0.581318  4.989280  5.610115  5.883223  6.294368   
mB           85.0  5.955067  0.654464  4.607072  5.483574  5.951272  6.362665   
mb        19776.0  4.931991  0.366626  3.139041  4.694206  4.912030  5.152664   

               max  
mag_ty

In [9]:
big = idn[idn["magnitude"] >= 4].copy()

# Wilayah paling aktif
print(big["location"].value_counts().head(10))

# Kategori kedalaman: dangkal / menengah / dalam
big["kelas_depth"] = pd.cut(big["depth"], [-1, 70, 300, 1000],
                            labels=["dangkal", "menengah", "dalam"])

print(big.groupby("kelas_depth", observed=True)["magnitude"].agg(["count", "mean"]))

location
Banda Sea                        5278
Northern Molucca Sea             2535
Minahassa Peninsula, Sulawesi    2231
Talaud Islands, Indonesia        2079
Southern Sumatra, Indonesia      2047
Halmahera, Indonesia             1912
Northern Sumatra, Indonesia      1556
Irian Jaya Region, Indonesia     1555
Java, Indonesia                  1411
South of Java, Indonesia         1302
Name: count, dtype: int64
             count      mean
kelas_depth                 
dangkal      24416  4.488203
menengah     10219  4.512658
dalam         1619  4.493006


In [10]:
semua = d["tahun"].value_counts().sort_index()
besar = idn[idn["magnitude"] >= 4]["tahun"].value_counts().sort_index()

banding = pd.DataFrame({"semua_gempa": semua, "magnitudo_4_ke_atas": besar})
print(banding)

       semua_gempa  magnitudo_4_ke_atas
tahun                                  
2008           239                   96
2009          4134                 1823
2010          7646                 2616
2011          6261                 1885
2012          8510                 2217
2013          5051                 1398
2014          7209                 1819
2015          7511                 1874
2016          7991                 1752
2017          8898                 1835
2018         12517                 2113
2019         13635                 2804
2020          9911                 2393
2021         12653                 2450
2022         12438                 2245
2023         13389                 2274
2024          9097                 1716
2025         21021                 1745
2026          8273                 1199


In [11]:
d["datetime"] = pd.to_datetime(d["datetime"], format="ISO8601", utc=True)
d["tahun"] = d["datetime"].dt.year

In [12]:
idn.to_csv("idn_clean.csv", index=False)
banding.to_csv("tren_per_tahun.csv")